In [25]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm


In [26]:
folders = [
# r"../Data/raw/saxophone_data",
# r"../Data/raw/piano_data",
# r"../Data/raw/guitar_data",
# r"../Data/raw/symphony_data",
r"../Data/raw/violin_data"
]

In [27]:
def extract_features(file_path):

    y, sr = librosa.load(file_path, sr=22050)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    # Tonnetz
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    tonnetz_mean = np.mean(tonnetz, axis=1) 

    # Spectral features
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    features = np.hstack([
        mfcc_mean,
        chroma_mean,
        tonnetz_mean,
        centroid,
        bandwidth,
        rolloff,
        zcr,
        tempo
    ])

    return features


In [28]:

data = []

for folder in folders:

    instrument = os.path.basename(folder)

    for file in tqdm(os.listdir(folder)):

        if file.endswith(".mp3"):

            path = os.path.join(folder, file)

            try:
                features = extract_features(path)

                row = {
                    "file_name": file,
                    "path": path,
                    "instrument": instrument
                }

                # MFCC
                for i in range(13):
                    row[f"mfcc_{i+1}"] = features[i]

                # Chroma
                chroma_labels = [
                    "C","Csharp","D","Dsharp","E","F",
                    "Fsharp","G","Gsharp","A","Asharp","B"
                ]

                for i, label in enumerate(chroma_labels):
                    row[f"chroma_{label}"] = features[13 + i]

                # Tonnetz
                for i in range(6):
                    row[f"tonnetz_{i+1}"] = features[25 + i]

                # Other features
                row["spectral_centroid"] = features[31]
                row["spectral_bandwidth"] = features[32]
                row["spectral_rolloff"] = features[33]
                row["zero_crossing_rate"] = features[34]
                row["tempo"] = features[35]

                data.append(row)

            except Exception as e:
                print("Error:", file, e)

df = pd.DataFrame(data)

100%|██████████| 189/189 [17:51<00:00,  5.67s/it]


In [29]:
df

,file_name,path,instrument,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,...,tonnetz_2,tonnetz_3,tonnetz_4,tonnetz_5,tonnetz_6,spectral_centroid,spectral_bandwidth,spectral_rolloff,zero_crossing_rate,tempo
0,vio_001_classic.mp3,../Data/raw/violin_data\vio_001_classic.mp3,violin_data,-305.353516,135.164825,54.055496,25.585520,11.411432,-0.643234,-2.686048,...,0.244834,0.037902,0.104002,-0.016143,-0.027718,1567.939060,1916.289793,3454.437445,0.062333,123.046875
1,vio_002_classic.mp3,../Data/raw/violin_data\vio_002_classic.mp3,violin_data,-302.237671,105.775986,25.653852,24.149714,20.070288,-3.616683,-4.809196,...,0.002237,0.004201,-0.273116,-0.010817,0.044238,1752.926146,2266.883701,3788.318759,0.053109,123.046875
2,vio_003_classic.mp3,../Data/raw/violin_data\vio_003_classic.mp3,violin_data,-261.376221,123.468384,36.117840,23.769503,10.132271,-1.288292,0.861046,...,0.168537,0.213862,0.040018,-0.007196,0.011841,1849.227574,2351.138635,4131.000020,0.065992,95.703125
3,vio_004_classic.mp3,../Data/raw/violin_data\vio_004_classic.mp3,violin_data,-277.222076,111.399239,42.291363,25.704132,22.623947,1.436186,-3.832210,...,-0.042024,0.026574,-0.242437,0.010491,-0.003960,1758.451868,2335.269663,3969.084333,0.052493,95.703125
4,vio_005_classic.mp3,../Data/raw/violin_data\vio_005_classic.mp3,violin_data,-247.332184,107.263184,15.898631,20.288799,8.064924,9.893385,1.515058,...,0.131372,-0.141021,-0.199758,-0.034311,-0.041125,1756.100882,2159.507100,3557.703487,0.060404,112.347147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,vio_185_classic.mp3,../Data/raw/violin_data\vio_185_classic.mp3,violin_data,-306.200439,161.109589,-3.546070,19.899668,-4.967171,1.811903,-3.230709,...,0.007963,-0.107320,0.181891,-0.026305,0.024830,1082.516008,1273.094602,1910.588447,0.059360,135.999178
185,vio_186_classic.mp3,../Data/raw/violin_data\vio_186_classic.mp3,violin_data,-308.671600,175.914276,0.206250,13.216725,4.190360,2.921061,-5.016991,...,-0.081555,0.294255,-0.076531,-0.003558,0.016581,848.538532,1034.481105,1389.041601,0.053623,89.102909
186,vio_187_classic.mp3,../Data/raw/violin_data\vio_187_classic.mp3,violin_data,-205.355377,108.501732,6.767164,30.287119,1.323946,-7.310257,-4.129194,...,-0.013364,-0.130866,0.152695,0.000758,-0.002004,1658.284959,1951.747919,3455.120774,0.072878,161.499023
187,vio_188_classic.mp3,../Data/raw/violin_data\vio_188_classic.mp3,violin_data,-291.343323,136.381287,2.176591,21.011280,2.703120,4.603218,-11.274522,...,-0.152799,0.085676,-0.065983,-0.016082,0.047215,1322.980891,1583.529065,2536.601328,0.067082,123.046875


In [30]:

df.to_csv("../Results/violin_feature_database.csv", index=False)

print("Feature extraction finished!")
print("Total files processed:", len(df))

Feature extraction finished!
Total files processed: 189
